# DHGCMDA Interactive Runner

Đây là Jupyter Notebook được tạo lại từ dự án DHGCMDA. File này sẽ giúp bạn chạy thử nghiệm (train & evaluate) mô hình dự đoán liên kết miRNA-Disease trực tiếp một cách tương tác.

Notebook này tái sử dụng các module có sẵn trong thư mục dự án (như `param.py`, `trainData.py`, `hetero_model.py`, `main_experiments_hetero1.py`) để đảm bảo tính nhất quán với code gốc.

## 1. Cài đặt và Import thư viện

In [ ]:
import os
import sys
import warnings
import torch
import torch.optim as optim
import numpy as np
import random

warnings.filterwarnings('ignore')

# Import các module từ dự án
from param import parameter_parser
from trainData import get_train_data
from hetero_model import HeterogenousGraphCLAMIR
from Calculate_Metrics import Metric_fun
from main_experiments_hetero1 import (
    SimplifiedMultiTypeAssociationLoss, 
    create_hetero_data_optimized, 
    constructHW_knn
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Thiết lập Random Seed
def seed_torch(seed=1234):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
seed_torch()

## 2. Cấu hình Tham số (Hyperparameters)

Khởi tạo và ghi đè các tham số cần thiết. Theo kết quả tối ưu nhất từ dự án (Plan M), chúng ta dùng cấu hình K_neigs=2 và predictor_mode='full_bilinear'.

In [ ]:
import argparse

# Tránh lỗi argparse trong Jupyter Notebook bằng cách parse mảng rỗng
sys.argv = [''] 
args = parameter_parser()
args.device = str(device)

# Cấu hình tối ưu (Plan M)
args.K_neigs = [2]
args.predictor_mode = 'full_bilinear'
args.dataset = 'v2.0_495m383D'
args.epoch = 50  # Chạy 50 epoch để demo nhanh (Mặc định là 650)
args.batch_size = 1000

print(f"Dataset: {args.dataset}")
print(f"Epochs: {args.epoch}")
print(f"Predictor Mode: {args.predictor_mode}")

## 3. Tải Dữ liệu

In [ ]:
print("Đang tải dữ liệu...")
train_data = get_train_data(args)
dis_sim = train_data[0].float().to(device)
mi_sim = train_data[1].float().to(device)
association_matrix = train_data[4].float().to(device)

print("Kích thước ma trận liên kết:", association_matrix.shape)
print("Số lượng liên kết dương (positive):", (association_matrix > 0).sum().item())

## 4. Khởi tạo Mô hình & Loss Function

In [ ]:
model = HeterogenousGraphCLAMIR(args).to(device)
optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
criterion = SimplifiedMultiTypeAssociationLoss(args, model).to(device)

print("Mô hình đã khởi tạo thành công!")

## 5. Chuẩn bị Dữ liệu Đồ thị

In [ ]:
# Tạo dữ liệu HeteroData (cho HGT)
hetero_data = create_hetero_data_optimized(train_data)

# Tạo siêu đồ thị Hypergraph (cho HGCN)
mi_H = constructHW_knn(train_data[1].cpu().numpy(), args.K_neigs, is_probH=False).to(device)
dis_H = constructHW_knn(train_data[0].cpu().numpy(), args.K_neigs, is_probH=False).to(device)

# Trích xuất tập index train (tất cả các liên kết)
pos_indices = torch.nonzero(association_matrix > 0, as_tuple=False)
neg_indices = torch.nonzero(association_matrix == 0, as_tuple=False)

# Lấy mẫu ngẫu nhiên tập Negative với tỷ lệ 1:1 với Positive
idx = torch.randperm(neg_indices.size(0))[:pos_indices.size(0)]
sampled_neg_indices = neg_indices[idx]

print(f"Positive samples: {len(pos_indices)}")
print(f"Negative samples: {len(sampled_neg_indices)}")

## 6. Huấn luyện (Training Loop)

In [ ]:
print("Bắt đầu huấn luyện...")

for epoch in range(1, args.epoch + 1):
    model.train()
    optimizer.zero_grad()
    
    # Forward Pass
    score, (z1_mi, z1_dis, z2_mi, z2_dis), (feat_mi, feat_dis), _ = model(
        hetero_data, mi_H, dis_H, mi_sim, dis_sim
    )
    
    # Compute Loss
    loss = criterion(pos_indices, sampled_neg_indices, score, association_matrix)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}/{args.epoch:03d} - Loss: {loss.item():.4f}")

print("Hoàn tất huấn luyện!")

## 7. Đánh giá nhanh (Evaluation)

Thực hiện tính toán các metric (AUC, AUPR, F1) trên tập dữ liệu. (Lưu ý: trong thực tế cần chạy K-fold cross-validation, đây chỉ là đánh giá nhanh trên tập vừa train).

In [ ]:
model.eval()
with torch.no_grad():
    score, _, _, _ = model(hetero_data, mi_H, dis_H, mi_sim, dis_sim)
    
    y_true_binary = (association_matrix > 0).float().cpu().numpy().flatten()
    y_pred_exist = score[:, :, 0].cpu().numpy().flatten()
    
    from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
    
    auc = roc_auc_score(y_true_binary, y_pred_exist)
    aupr = average_precision_score(y_true_binary, y_pred_exist)
    
    # Tính F1 với threshold 0.5
    y_pred_binary = (y_pred_exist > 0.5).astype(int)
    f1 = f1_score(y_true_binary, y_pred_binary)
    
    print("--- Kết quả Đánh giá Nhanh ---")
    print(f"AUC:  {auc:.4f}")
    print(f"AUPR: {aupr:.4f}")
    print(f"F1:   {f1:.4f}")